In [7]:
from ultralytics import YOLO

# Load a pretrained classification model as a starting point
# (use yolov8n-cls.pt or yolov11n-cls.pt depending on what you have)
model = YOLO("yolov8n-cls.pt")   # or "yolov11n-cls.pt" if available

# Train on your labeled crops
model.train(
    data="number_crops_split_new2",  # 👈 directory, NOT YAML
    epochs=20,
    imgsz=64,
    batch=16
)

New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=number_crops_split_new2, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optim

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x138c3f8b0>
curves: []
curves_results: []
fitness: 0.9090909063816071
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8181818127632141, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9090909063816071}
save_dir: PosixPath('/Users/lucaspedemonte/Documents/VSCode/NFL_Play_Outcome_Classifier/runs/classify/train4')
speed: {'preprocess': 0.0004706590950511533, 'inference': 1.564942227270711, 'loss': 3.406817292918938e-05, 'postprocess': 5.6818180382833816e-05}
task: 'classify'
top1: 0.8181818127632141
top5: 1.0

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import numpy as np

# ============================
# CONFIG (update your paths)
# ============================
FIELD_MODEL_WEIGHTS = "runs/detect/football_yolo11n2/weights/best.pt"    # detects "number"
NUMBER_MODEL_WEIGHTS = "runs/classify/train4/weights/best.pt" # classifier you just trained
TEST_IMAGE_PATH = "data/Football Player Detection.v7i.yolov11/test/images/57503_004156_Sideline_frame0643_jpg.rf.0fe4c096792e2cb8fa6a9d0f2e8bdacd.jpg"                  # any NFL frame

# ============================
# LOAD MODELS
# ============================
field_model = YOLO(FIELD_MODEL_WEIGHTS)
number_model = YOLO(NUMBER_MODEL_WEIGHTS)

print("Field model classes:", field_model.names)
print("Number model classes:", number_model.names)

# ============================
# DETECT NUMBERS IN THE IMAGE
# ============================
img = cv2.imread(TEST_IMAGE_PATH)
if img is None:
    raise FileNotFoundError(TEST_IMAGE_PATH)

results = field_model(img)[0]

# Extract detections
def extract_detections(results):
    dets = []
    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        dets.append({
            "class_id": cls_id,
            "class_name": results.names[cls_id],
            "confidence": conf,
            "bbox": (x1, y1, x2, y2),
        })
    return dets

dets = extract_detections(results)

# Filter only "number"
number_dets = [d for d in dets if "number" in d["class_name"].lower()]
print(f"Found {len(number_dets)} number detections.")

# ============================
# CLASSIFY EACH NUMBER CROP
# ============================
predictions = []

for i, det in enumerate(number_dets):
    x1, y1, x2, y2 = map(int, det["bbox"])
    crop_bgr = img[y1:y2, x1:x2]

    if crop_bgr.size == 0:
        continue

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    # classify with number model
    r = number_model(crop_rgb)[0]
    cls_id = int(r.probs.top1)
    cls_name = number_model.names[cls_id]
    conf = float(r.probs.top1conf)

    predictions.append({
        "bbox": det["bbox"],
        "predicted_number": cls_name,
        "confidence": conf,
    })

    print(f"Detection {i}: Predicted {cls_name} ({conf:.2f})")

# ============================
# VISUALIZE RESULT
# ============================
# ============================
# VISUALIZE + SAVE OUTPUT
# ============================

# Make a copy to draw on
vis = img.copy()

for pred in predictions:
    x1, y1, x2, y2 = map(int, pred["bbox"])
    number_label = pred["predicted_number"]
    conf = pred["confidence"]

    label_text = f"{number_label} ({conf:.2f})"

    # Draw bounding box
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)

    # Draw text label
    cv2.putText(
        vis,
        label_text,
        (x1, max(0, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

# Display in notebook
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Number Detection + Classification Overlay")
plt.show()

# ============================
# SAVE TO FILE
# ============================

output_path = "classified_numbers_overlay2.jpg"
cv2.imwrite(output_path, vis)
print(f"Saved annotated image to: {output_path}")

Field model classes: {0: 'ball', 1: 'five_hash', 2: 'hash', 3: 'marker', 4: 'number', 5: 'player', 6: 'pylon', 7: 'ref'}
Number model classes: {0: '10', 1: '20', 2: '30', 3: '40', 4: '50'}

0: 736x1280 11 five_hashs, 55 hashs, 2 markers, 9 numbers, 74.8ms
Speed: 5.2ms preprocess, 74.8ms inference, 1.1ms postprocess per image at shape (1, 3, 736, 1280)
Found 9 number detections.

0: 64x64 30 0.84, 50 0.07, 20 0.07, 10 0.01, 40 0.01, 1.5ms
Speed: 0.6ms preprocess, 1.5ms inference, 0.0ms postprocess per image at shape (1, 3, 64, 64)
Detection 0: Predicted 30 (0.84)

0: 64x64 40 0.52, 50 0.44, 30 0.02, 20 0.01, 10 0.01, 1.2ms
Speed: 0.6ms preprocess, 1.2ms inference, 0.0ms postprocess per image at shape (1, 3, 64, 64)
Detection 1: Predicted 40 (0.52)

0: 64x64 20 0.79, 10 0.12, 30 0.09, 50 0.00, 40 0.00, 1.5ms
Speed: 0.4ms preprocess, 1.5ms inference, 0.0ms postprocess per image at shape (1, 3, 64, 64)
Detection 2: Predicted 20 (0.79)

0: 64x64 30 0.36, 20 0.25, 10 0.21, 40 0.09, 50 0.09, 

<Figure size 1200x800 with 1 Axes>